# Notebook 03: Building the ReAct Agent

Implementing the Reason-Act-Observe loop with OpenAI function calling.

## 1. Setup

In [1]:
import os
import json
import random
from typing import Dict, List
from dotenv import load_dotenv
from openai import OpenAI
import boto3

load_dotenv()

LLM_BASE_URL = os.getenv('LLM_BASE_URL')
LLM_API_KEY = os.getenv('LLM_API_KEY')
LLM_MODEL = os.getenv('LLM_MODEL', 'gpt-4o')
KNOWLEDGE_BASE_ID = os.getenv('KNOWLEDGE_BASE_ID')

llm_client = OpenAI(base_url=LLM_BASE_URL, api_key=LLM_API_KEY)
bedrock_agent = boto3.client('bedrock-agent-runtime', region_name='us-east-1')

print('[OK] Configuration loaded')

[OK] Configuration loaded


## 2. Tools (from Notebook 02)

In [2]:
# Mock data
MOCK_ORDERS = {
    "ORD-12345": {"status": "shipped", "carrier": "FedEx", "estimated_delivery": "2026-02-03", 
                  "items": ["Blue iPhone 15 Case"], "destination": "New York, NY"},
    "ORD-67890": {"status": "processing", "estimated_delivery": "2026-02-05", 
                  "items": ["Wireless Earbuds"], "destination": "Miami, FL"},
    "ORD-11111": {"status": "delivered", "carrier": "UPS", "delivered_date": "2026-01-28", 
                  "items": ["Laptop Stand"], "destination": "Los Angeles, CA"}
}

MOCK_WEATHER = {
    "miami": {"location": "Miami, FL", "condition": "Hurricane Warning", "estimated_delay_days": 3},
    "new york": {"location": "New York, NY", "condition": "Clear", "estimated_delay_days": 0},
    "chicago": {"location": "Chicago, IL", "condition": "Winter Storm", "estimated_delay_days": 1}
}

MOCK_INVENTORY = {
    "iphone 15 case": {"blue": {"in_stock": True, "quantity": 42, "price": 29.99}, 
                       "black": {"in_stock": False, "quantity": 0}},
    "airpods pro": {"default": {"in_stock": True, "quantity": 120, "price": 199.00}}
}

# Tool implementations
def search_knowledge_base(query: str) -> Dict:
    try:
        response = bedrock_agent.retrieve(
            knowledgeBaseId=KNOWLEDGE_BASE_ID, retrievalQuery={'text': query},
            retrievalConfiguration={'vectorSearchConfiguration': {'numberOfResults': 3}}
        )
        results = [{'content': r['content']['text']} for r in response.get('retrievalResults', [])]
        return {'success': True, 'results': results}
    except Exception as e:
        return {'success': False, 'error': str(e)}

def check_order_status(order_id: str) -> Dict:
    order_id = order_id.upper().strip()
    if not order_id.startswith("ORD-"): order_id = f"ORD-{order_id}"
    if order_id in MOCK_ORDERS:
        return {'success': True, 'order_id': order_id, **MOCK_ORDERS[order_id]}
    return {'success': False, 'error': f'Order {order_id} not found'}

def get_weather_alerts(location: str) -> Dict:
    key = location.lower().split(',')[0].strip()
    if key in MOCK_WEATHER:
        return {'success': True, **MOCK_WEATHER[key]}
    return {'success': True, 'location': location, 'condition': 'Clear', 'estimated_delay_days': 0}

def check_inventory(product_name: str, color: str = None) -> Dict:
    key = product_name.lower().strip()
    for pkey in MOCK_INVENTORY:
        if pkey in key or key in pkey:
            data = MOCK_INVENTORY[pkey]
            if color and color.lower() in data:
                return {'success': True, 'product': pkey.title(), **data[color.lower()]}
            return {'success': True, 'product': pkey.title(), 'variants': data}
    return {'success': False, 'error': 'Product not found'}

def create_return_request(order_id: str, reason: str) -> Dict:
    order_id = order_id.upper().strip()
    if not order_id.startswith("ORD-"): order_id = f"ORD-{order_id}"
    if order_id not in MOCK_ORDERS:
        return {'success': False, 'error': 'Order not found'}
    return_id = f"RET-{random.randint(10000, 99999)}"
    return {'success': True, 'return_id': return_id, 'order_id': order_id}

print('[OK] Tools defined')

[OK] Tools defined


## 3. Tool Schemas

In [3]:
TOOLS = [
    {"type": "function", "function": {"name": "search_knowledge_base", 
        "description": "Search company knowledge base for policies, FAQs, shipping info, and general questions.",
        "parameters": {"type": "object", "properties": {"query": {"type": "string", "description": "The search query"}}, "required": ["query"]}}},
    {"type": "function", "function": {"name": "check_order_status", 
        "description": "Check order status, tracking, and delivery ETA. Requires order ID.",
        "parameters": {"type": "object", "properties": {"order_id": {"type": "string", "description": "Order ID (e.g., ORD-12345)"}}, "required": ["order_id"]}}},
    {"type": "function", "function": {"name": "get_weather_alerts", 
        "description": "Check weather alerts and shipping delays for a location. Use when customer asks about weather impact on deliveries or delays to a city.",
        "parameters": {"type": "object", "properties": {"location": {"type": "string", "description": "City name (e.g., Miami, New York)"}}, "required": ["location"]}}},
    {"type": "function", "function": {"name": "check_inventory", 
        "description": "Check if a product is in stock and available quantities.",
        "parameters": {"type": "object", "properties": {"product_name": {"type": "string", "description": "Product name"}, "color": {"type": "string", "description": "Color variant (optional)"}}, "required": ["product_name"]}}},
    {"type": "function", "function": {"name": "create_return_request", 
        "description": "Create a return request for an order. Customer must provide order ID and reason.",
        "parameters": {"type": "object", "properties": {"order_id": {"type": "string", "description": "Order ID to return"}, "reason": {"type": "string", "description": "Reason for return"}}, "required": ["order_id", "reason"]}}}
]

TOOL_FUNCTIONS = {
    "search_knowledge_base": search_knowledge_base,
    "check_order_status": check_order_status,
    "get_weather_alerts": get_weather_alerts,
    "check_inventory": check_inventory,
    "create_return_request": create_return_request
}

print(f'[OK] {len(TOOLS)} tool schemas defined')

[OK] 5 tool schemas defined


## 4. ReAct Agent Class

In [5]:
class ReActAgent:
    """ReAct Agent: Reason -> Act -> Observe -> Repeat"""
    
    def __init__(self, model: str = None, max_iterations: int = 5, verbose: bool = True):
        self.model = model or LLM_MODEL
        self.max_iterations = max_iterations
        self.verbose = verbose
        self.system_prompt = """You are a helpful e-commerce customer service agent.

Available tools:
- search_knowledge_base: For policies, FAQs, general questions
- check_order_status: For order tracking (requires order ID)
- get_weather_alerts: For weather-related shipping delays (just needs city name)
- check_inventory: For stock availability
- create_return_request: To process returns (requires order ID and reason)

IMPORTANT: Be proactive - use tools immediately when relevant. For weather/delay questions about a city, call get_weather_alerts directly with the city name."""
    
    def _log(self, msg):
        if self.verbose: print(f"[Agent] {msg}")
    
    def _execute_tool(self, name: str, args: Dict) -> str:
        self._log(f"Calling {name}({args})")
        result = TOOL_FUNCTIONS[name](**args) if name in TOOL_FUNCTIONS else {"error": "Unknown tool"}
        return json.dumps(result)
    
    def run(self, query: str) -> Dict:
        self._log(f"Query: {query}")
        messages = [{"role": "system", "content": self.system_prompt}, {"role": "user", "content": query}]
        tools_used = []
        
        for iteration in range(1, self.max_iterations + 1):
            self._log(f"Iteration {iteration}")
            
            response = llm_client.chat.completions.create(
                model=self.model, messages=messages, tools=TOOLS, tool_choice="auto"
            )
            msg = response.choices[0].message
            
            if msg.tool_calls:
                messages.append({"role": "assistant", "content": msg.content,
                    "tool_calls": [{"id": tc.id, "type": "function", 
                                   "function": {"name": tc.function.name, "arguments": tc.function.arguments}}
                                  for tc in msg.tool_calls]})
                
                for tc in msg.tool_calls:
                    result = self._execute_tool(tc.function.name, json.loads(tc.function.arguments))
                    tools_used.append({"tool": tc.function.name, "args": json.loads(tc.function.arguments)})
                    messages.append({"role": "tool", "tool_call_id": tc.id, "content": result})
            else:
                self._log("Final response")
                return {"response": msg.content, "tools_used": tools_used, "iterations": iteration}
        
        return {"response": "Max iterations reached", "tools_used": tools_used, "iterations": self.max_iterations}

agent = ReActAgent(verbose=True)
print('[OK] ReAct Agent created')

[OK] ReAct Agent created


## 5. Test: Knowledge Base Query

In [6]:
result = agent.run("What is your return policy?")
print("\nResponse:", result['response'])
print("Tools:", [t['tool'] for t in result['tools_used']])

[Agent] Query: What is your return policy?
[Agent] Iteration 1
[Agent] Calling search_knowledge_base({'query': 'return policy'})
[Agent] Iteration 2
[Agent] Final response

Response: Our return policy allows you to return products within 30 days of purchase for a full refund, provided they are in their original condition and packaging. Here are some additional details:

- **Eligibility for Returns**: Items can be returned within 30 days of delivery. They must be in their original condition with all tags attached.
- **Damaged or Defective Items**: These qualify for a full refund with no restocking fee.
- **Change of Mind**: If you're returning an item for reasons other than damage or defect, a 15% restocking fee will be applied to the refund amount.
- **Improper Use**: Items damaged due to improper use may not be eligible for return.

For more detailed instructions, please refer to our Returns page or contact our customer support team.
Tools: ['search_knowledge_base']


## 6. Test: Order Status (Where Baseline RAG Failed)

In [7]:
result = agent.run("What is the status of my order #ORD-12345?")
print("\nResponse:", result['response'])
print("Tools:", [t['tool'] for t in result['tools_used']])

[Agent] Query: What is the status of my order #ORD-12345?
[Agent] Iteration 1
[Agent] Calling check_order_status({'order_id': 'ORD-12345'})
[Agent] Iteration 2
[Agent] Final response

Response: Your order #ORD-12345 has been shipped via FedEx. It contains a Blue iPhone 15 Case and is estimated to be delivered by February 3, 2026, to New York, NY. If you need any further assistance, feel free to ask!
Tools: ['check_order_status']


## 7. Test: Weather Delay

In [8]:
result = agent.run("Will my package to Miami be delayed? I heard there's bad weather.")
print("\nResponse:", result['response'])
print("Tools:", [t['tool'] for t in result['tools_used']])

[Agent] Query: Will my package to Miami be delayed? I heard there's bad weather.
[Agent] Iteration 1
[Agent] Calling get_weather_alerts({'location': 'Miami'})
[Agent] Iteration 2
[Agent] Final response

Response: There is currently a Hurricane Warning in Miami, which is likely to cause shipping delays. Your package might be delayed by approximately 3 days due to these weather conditions.
Tools: ['get_weather_alerts']


## 8. Test: Inventory Check

In [9]:
result = agent.run("Is the blue iPhone 15 case in stock?")
print("\nResponse:", result['response'])
print("Tools:", [t['tool'] for t in result['tools_used']])

[Agent] Query: Is the blue iPhone 15 case in stock?
[Agent] Iteration 1
[Agent] Calling check_inventory({'product_name': 'iPhone 15 case', 'color': 'blue'})
[Agent] Iteration 2
[Agent] Final response

Response: The blue iPhone 15 case is in stock, with 42 units available. The price is $29.99. Would you like to purchase one?
Tools: ['check_inventory']


## 9. Test: Create Return (Action)

In [10]:
result = agent.run("I want to return order #ORD-12345 because the item is defective.")
print("\nResponse:", result['response'])
print("Tools:", [t['tool'] for t in result['tools_used']])

[Agent] Query: I want to return order #ORD-12345 because the item is defective.
[Agent] Iteration 1
[Agent] Calling create_return_request({'order_id': 'ORD-12345', 'reason': 'Defective item'})
[Agent] Iteration 2
[Agent] Final response

Response: Your return request for order #ORD-12345 has been successfully processed due to a defective item. Your return ID is RET-21912. You will receive further instructions on returning the item shortly.
Tools: ['create_return_request']


## 10. Test: Multi-Tool Query

In [12]:
result = agent.run("What's the status of order ORD-67890 and will it be delayed due to weather in the destination?")
print("\nResponse:", result['response'])
print("Tools:", [t['tool'] for t in result['tools_used']])

[Agent] Query: What's the status of order ORD-67890 and will it be delayed due to weather in the destination?
[Agent] Iteration 1
[Agent] Calling check_order_status({'order_id': 'ORD-67890'})
[Agent] Iteration 2
[Agent] Calling get_weather_alerts({'location': 'Miami, FL'})
[Agent] Iteration 3
[Agent] Final response

Response: The current status of your order (ORD-67890) is "processing," with an estimated delivery date of February 5, 2026. However, there's a Hurricane Warning in Miami, FL, which could delay your delivery by approximately 3 days. Please keep an eye on any updates, and feel free to reach out if you have further questions or need assistance!
Tools: ['check_order_status', 'get_weather_alerts']


## 11. Comparison: Baseline RAG vs Agentic RAG

In [13]:
def baseline_rag(query: str) -> str:
    response = bedrock_agent.retrieve(
        knowledgeBaseId=KNOWLEDGE_BASE_ID, retrievalQuery={'text': query},
        retrievalConfiguration={'vectorSearchConfiguration': {'numberOfResults': 3}}
    )
    context = "\n".join([r['content']['text'] for r in response.get('retrievalResults', [])])
    llm_response = llm_client.chat.completions.create(
        model=LLM_MODEL,
        messages=[{"role": "system", "content": "Answer based on context only."},
                  {"role": "user", "content": f"Context:\n{context}\n\nQuestion: {query}"}]
    )
    return llm_response.choices[0].message.content

test_query = "What is the status of order #ORD-12345?"

print("=" * 60)
print(f"Query: {test_query}")
print("=" * 60)

print("\n[BASELINE RAG]")
print(baseline_rag(test_query))

print("\n[AGENTIC RAG]")
quiet_agent = ReActAgent(verbose=False)
result = quiet_agent.run(test_query)
print(result['response'])
print(f"(Tools: {[t['tool'] for t in result['tools_used']]})")

Query: What is the status of order #ORD-12345?

[BASELINE RAG]
I'm sorry, I don't have access to specific order information. Please track your order by logging into your account and checking the 'Order History' section or using the tracking number sent to your email.

[AGENTIC RAG]
Your order #ORD-12345 has been shipped via FedEx and is estimated to be delivered on February 3, 2026. The order contains a Blue iPhone 15 Case and is being delivered to New York, NY.
(Tools: ['check_order_status'])


## Summary

Built a ReAct agent that:
1. Receives query
2. Reasons about which tools to use
3. Executes tools via OpenAI function calling
4. Observes results
5. Repeats or responds

**Next:** Build complete chatbot with conversation memory.